#Project: 멋진 챗봇 만들기



### **프로젝트 목적**
---
본 프로젝트는 **17장에서 구현한 Transformer 번역 모델을 그대로 활용하여 한국어 챗봇을 구현**하는 것을 목표로 합니다.
번역기와 챗봇은 모두 '입력 시퀀스를 받아 출력 시퀀스를 생성한다'는 점에서 본질적으로 같은 구조(Seq2Seq)를 공유합니다. 다만 챗봇은 **소스와 타겟이 같은 언어(한국어-한국어)**라는 점, 그리고 데이터 규모가 훨씬 작다는 점(약 1만 개)이 다릅니다.

이번 프로젝트를 통해 다음을 확인합니다.
- 동일한 Transformer 아키텍처가 번역뿐 아니라 **대화형 응답 생성(챗봇)**에도 적용될 수 있는가
- 적은 데이터로 학습할 때 발생하는 **과적합 문제를 Augmentation과 하이퍼파라미터 튜닝으로 어떻게 완화**하는가
- 생성된 답변의 품질을 **정성적(직접 확인) + 정량적(BLEU Score)**으로 어떻게 평가하는가

<br>

### **목차**
---
- **Step 1. 데이터 다운로드**: 한국어 챗봇 데이터(`ChatbotData.csv`)를 불러와 `questions`, `answers`로 분리
- **Step 2. 데이터 정제**: `preprocess_sentence()` 구현 — 영문 소문자화, 정규식으로 한글/숫자/주요 특수문자 외 제거
- **Step 3. 데이터 토큰화**: `build_corpus()` 구현 — KoNLPy `mecab.morphs` 형태소 분석, 길이 초과/중복 문장 제외 → `que_corpus`, `ans_corpus`
- **Step 4. Augmentation**: 사전 훈련된 한국어 Word2Vec(`ko.bin`)으로 `lexical_sub()` 적용, 전체 데이터를 원본의 약 3배로 증강
- **Step 5. 데이터 벡터화**: `ans_corpus`에 `<start>`/`<end>` 토큰 추가 → 단어 사전 구축 및 `enc_train`, `dec_train` 벡터화
- **Step 6. 훈련하기**: Transformer 모델 재사용, 과적합 방지를 위한 하이퍼파라미터 튜닝 → 예문 4개에 대한 답변 생성
- **Step 7. 성능 측정하기**: 정성적 평가(예문 응답 확인) + `calculate_bleu()`로 BLEU Score 정량 평가

<br>

### **평가 루브릭**
---
|평가문항|상세기준|
|-|-|
|1. 챗봇 훈련데이터 전처리가 체계적으로 진행되었는가?|전처리와 augmentation이 적절히 수행되어 3만개 가량의 훈련데이터셋이 구축되었다.|
|2. Transformer 챗봇 모델이 과적합을 피해 안정적으로 훈련되었는가?|과적합을 피할 수 있는 하이퍼파라미터 셋이 적절히 제시되었다.|
|3. 챗봇이 사용자의 질문에 그럴듯한 형태로 답하는 사례가 있는가?|주어진 예문을 포함하여 챗봇에 던진 질문에 적절히 답하는 사례가 제출되었다.|



### **단계별 핵심 내용 및 주의사항**
---

**Step 1. 데이터 다운로드**
- 핵심: `ChatbotData.csv`(songys/Chatbot_data)를 `pandas.read_csv()`로 읽어 `Q`, `A` 컬럼을 각각 `questions`, `answers`로 분리
- 주의:
  - 이 데이터셋에는 감정 라벨(`label`) 컬럼이 함께 있음 — 이번 프로젝트에서는 사용하지 않으므로 Q/A만 정확히 골라낼 것
  - 인코딩은 UTF-8 기준이지만, 로컬/Colab 환경에 따라 경로·인코딩 에러가 날 수 있으니 `encoding='utf-8'` 명시 권장
  - 결측치(NaN) 여부 확인

**Step 2. 데이터 정제**
- 핵심: `preprocess_sentence()`로 소문자화 + 정규식으로 한글/영문/숫자/주요 문장부호 외 제거
- 주의:
  - 문장부호를 과도하게 제거하면 `?`, `!` 같은 의미 있는 정보가 사라져 챗봇 답변의 어투/뉘앙스가 손실될 수 있음 → 정규식 패턴에 남길 문자 신중히 결정
  - 이번 토크나이저(mecab)가 띄어쓰기·문장부호 처리를 어느 정도 지원하므로, 번역기 프로젝트 때처럼 공백 삽입 로직까지 넣을 필요는 없음
  - 정제 후 빈 문자열이 되는 문장이 있는지 확인 (있다면 Step 3에서 필터링 필요)

**Step 3. 데이터 토큰화**
- 핵심: `build_corpus()`로 `mecab.morphs` 형태소 분석 → 길이 제한 초과/중복 문장 제거 → `que_corpus`, `ans_corpus` 생성
- 주의:
  - **KoNLPy + mecab-ko 설치 이슈**가 가장 흔한 문제 — 로컬/Colab 환경별 설치 스크립트가 다르므로 미리 정상 동작 확인
  - 중복 제거는 **소스(질문)와 타겟(답변)을 각각 독립적으로 검사**하되, 두 리스트의 인덱스(쌍) 정합성이 깨지지 않도록 유의 (한쪽만 걸러내면 짝이 어긋남)
  - 길이 제한 기준을 너무 짧게 잡으면 데이터가 급격히 줄어들 수 있음 — 원본이 1만 개 내외로 이미 적은 편이라 필터링 강도 조절 필요

**Step 4. Augmentation (데이터 증강)**
- 핵심: 사전 훈련된 한국어 Word2Vec(`ko.bin`)으로 `lexical_sub()` 적용 → `(Aug 질문, 원본 답변)` + `(원본 질문, Aug 답변)` 조합으로 전체 데이터를 원본의 약 3배로 증강
- 주의:
  - `ko.bin` 파일이 용량이 크고 외부(Google Drive) 다운로드라 실행 환경에 따라 시간이 오래 걸리거나 실패할 수 있음
  - 임베딩에 없는 단어(OOV)는 치환 대상에서 제외하는 예외처리 필요
  - 과도한 치환 비율은 원문 의미를 왜곡시켜 오히려 노이즈로 작용할 수 있음 — 치환 강도(비율) 조절
  - 최종 데이터 규모가 "약 3만 개"에 도달하는지 루브릭 기준으로 반드시 확인

**Step 5. 데이터 벡터화**
- 핵심: `ans_corpus`에 `<start>`/`<end>` 토큰 추가 → `que_corpus`와 합쳐 **하나의 통합 단어 사전** 구축 → `enc_train`, `dec_train` 벡터화
- 주의:
  - 챗봇은 소스·타겟이 **같은 언어**이므로 사전을 통합하고 임베딩을 공유(`shared=True`)하는 것이 유리 — 번역기 프로젝트와 다른 부분
  - `MAX_LEN`(패딩 길이) 설정 시 Step 3에서 정한 길이 제한과 일관성 유지
  - 토큰 ID 체계(pad/bos/eos/unk id)를 이전 번역기 프로젝트와 동일하게 맞춰야 기존 Transformer 코드 재사용 시 오류가 없음

**Step 6. 훈련하기**
- 핵심: 17장에서 만든 `Transformer` 재사용 + 하이퍼파라미터 튜닝(레이어 수, d_model, d_ff, dropout, warmup steps 등) → 4개 예문에 대한 답변 생성
- 주의:
  - **데이터가 3만 개 내외로 작아 과적합이 매우 쉽게 발생** — 레이어 수/모델 크기를 줄이고 dropout을 높이는 등 번역기보다 가벼운 세팅 필요
  - 매 epoch마다 예문 답변을 직접 확인하며, 답변이 그럴듯해지는 시점의 하이퍼파라미터를 기록해 둘 것 (재현을 위해)
  - GPU 메모리/세션 제한(Colab) 고려해 배치 크기 조절
  - 랜덤 시드 고정 여부에 따라 결과가 달라질 수 있음을 인지

**Step 7. 성능 측정하기**
- 핵심: 정성적 평가(예문 응답 직접 확인) + `calculate_bleu()`로 BLEU Score 정량 평가
- 주의:
  - 챗봇 답변은 정답이 하나로 고정되지 않는 특성상(동일 질문에도 다양한 적절한 답 존재) **BLEU Score가 낮게 나올 수 있음** — 번역 성능처럼 절대적 지표로 해석하지 말고 참고 지표로 활용
  - `SmoothingFunction()`을 적용해 짧은 문장에서 BLEU가 0으로 튀는 문제 보정
  - 정성 평가(예문 4개)와 정량 평가(BLEU)를 함께 제시해야 루브릭 3번 항목을 충족

<br>

### **⚠️ 전체 공통 주의사항**
---
- 원본 데이터가 1만 개 내외로 작기 때문에, **Step 4 증강 → Step 6 튜닝**의 흐름이 전체 프로젝트의 성패를 좌우함
- 번역기 프로젝트(17장)의 코드를 최대한 재사용하되, **소스/타겟이 같은 언어**라는 챗봇 특성(사전 공유, 벡터화 방식)을 반영해야 함
- 각 Step의 산출물(`questions`/`answers`, `que_corpus`/`ans_corpus`, `enc_train`/`dec_train`)이 다음 Step 입력으로 그대로 이어지므로, 중간에 변수명을 바꾸지 않도록 주의



## **Step 1. 데이터 다운로드**
---
[songys/Chatbot_data](https://github.com/songys/Chatbot_data) 저장소의 `ChatbotData.csv`를 받아 `questions`, `answers`로 분리합니다.


In [14]:
# Step 1-1. 챗봇 데이터셋 다운로드
# songys/Chatbot_data 저장소가 제공하는 한국어 챗봇 질의응답 데이터(ChatbotData.csv)를 받아옵니다.
# 컬럼 구성: Q(질문), A(답변), label(감정 라벨: 0=일상다반사, 1=이별(부정), 2=사랑(긍정))
#   -> label은 이번 프로젝트(번역기 구조 재사용)에서는 사용하지 않습니다.
!wget -q https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv -O ChatbotData.csv

print("다운로드 완료: ChatbotData.csv")


다운로드 완료: ChatbotData.csv


In [15]:
import pandas as pd

# Step 1-2. 데이터 불러오기
# csv 파일을 판다스 DataFrame으로 읽어옵니다. (인코딩은 UTF-8 기준)
data = pd.read_csv("ChatbotData.csv", encoding="utf-8")

# 전체 크기와 컬럼 구성을 먼저 확인합니다.
print("데이터 shape:", data.shape)
print("컬럼:", list(data.columns))
data.head()


데이터 shape: (11823, 3)
컬럼: ['Q', 'A', 'label']


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [16]:
# Step 1-3. 질문(questions) / 답변(answers) 분리
# label 컬럼은 사용하지 않으므로 Q, A 컬럼만 리스트로 추출합니다.
questions = data['Q'].tolist()
answers = data['A'].tolist()

# 질문-답변 쌍이 한쪽만 밀려서 어긋나지 않았는지, 개수가 일치하는지 확인합니다.
assert len(questions) == len(answers), "질문과 답변의 개수가 일치하지 않습니다!"

# 결측치(NaN) 여부를 확인합니다. (있다면 이후 정제 단계에서 처리 필요)
print(f"전체 데이터 개수: {len(questions)}개")
print(f"결측치 개수 - questions: {data['Q'].isnull().sum()}, answers: {data['A'].isnull().sum()}")

# 샘플을 몇 개 출력해 데이터가 의도대로 분리되었는지 눈으로 확인합니다.
for i in range(3):
    print(f"Q: {questions[i]}  ->  A: {answers[i]}")


전체 데이터 개수: 11823개
결측치 개수 - questions: 0, answers: 0
Q: 12시 땡!  ->  A: 하루가 또 가네요.
Q: 1지망 학교 떨어졌어  ->  A: 위로해 드립니다.
Q: 3박4일 놀러가고 싶다  ->  A: 여행은 언제나 좋죠.



## **Step 2. 데이터 정제**
---
`preprocess_sentence()` 함수를 구현해 문장을 정제합니다.
1. 영문자는 모두 소문자로 변환
2. 한글/영문/숫자/주요 문장부호 외 특수문자는 정규식으로 제거
(문장부호 양옆 공백 삽입 등은 mecab 토크나이저가 지원하므로 생략)


In [17]:
import re

def preprocess_sentence(sentence):
    # 1. 영문자를 모두 소문자로 변환합니다. (한글에는 영향 없음)
    sentence = sentence.lower()

    # 2. 한글(가-힣), 영문(a-z), 숫자(0-9), 주요 문장부호(? . ! ,)를 제외한
    #    나머지 특수문자는 모두 공백으로 치환합니다.
    #    -> 문장부호 양옆에 공백을 넣어주는 처리는 mecab 토크나이저가 대신 해주므로 생략합니다.
    sentence = re.sub(r"[^0-9a-zA-Z가-힣?.!,]+", " ", sentence)

    # 3. 위 과정에서 생긴 연속 공백을 하나로 합칩니다.
    sentence = re.sub(r"\s+", " ", sentence)

    # 4. 문자열 양 끝의 불필요한 공백을 제거합니다.
    sentence = sentence.strip()

    return sentence

print("슝=3")  # 정상적으로 함수가 정의되었는지 확인


슝=3


In [18]:
# Step 2 적용. 전체 질문/답변 데이터를 정제합니다.
# (원본은 비교용으로 남겨두기 위해 별도 변수에 백업)
raw_questions, raw_answers = questions, answers

questions = [preprocess_sentence(q) for q in raw_questions]
answers = [preprocess_sentence(a) for a in raw_answers]

# 정제 전/후 결과를 몇 개 샘플로 비교합니다.
for i in range(3):
    print(f"[원본] {raw_questions[i]}")
    print(f"[정제] {questions[i]}")
    print()

# 정제 후 빈 문자열이 된 데이터가 있는지 확인합니다.
# (특수문자로만 이루어진 문장이 있었다면 빈 문자열이 될 수 있고, Step 3에서 걸러내야 합니다.)
empty_q = sum(1 for q in questions if q == "")
empty_a = sum(1 for a in answers if a == "")
print(f"정제 후 빈 문자열 개수 - questions: {empty_q}, answers: {empty_a}")


[원본] 12시 땡!
[정제] 12시 땡!

[원본] 1지망 학교 떨어졌어
[정제] 1지망 학교 떨어졌어

[원본] 3박4일 놀러가고 싶다
[정제] 3박4일 놀러가고 싶다

정제 후 빈 문자열 개수 - questions: 0, answers: 0



## **Step 3. 데이터 토큰화**
---
KoNLPy의 `mecab.morphs`로 형태소 분석 후, 길이 제한을 초과하거나 중복되는 문장을 제외해 `que_corpus`, `ans_corpus`를 만듭니다.


In [19]:
# mecab-ko 설치 (Colab 환경 기준)
# 기존에 많이 쓰이던 konlpy.tag.Mecab + mecab-python(0.996) 조합은 최신 Colab의
# Python/pip 버전에서 "python setup.py egg_info did not run successfully" 오류가 발생하며 설치가 실패합니다.
# (mecab-python 0.996이 아주 오래된 패키지라 최신 setuptools와 호환되지 않기 때문입니다.)
#
# 대신 사전 컴파일된 mecab-ko를 wheel에 포함해 별도 설치 스크립트 없이 pip만으로 설치되는
# python-mecab-ko 패키지를 사용합니다. (API가 KoNLPy의 Mecab과 거의 동일 - morphs() 그대로 사용 가능)
!pip install -q python-mecab-ko


In [20]:
from mecab import MeCab  # python-mecab-ko 패키지 (konlpy.tag.Mecab과 동일하게 morphs() 사용 가능)

mecab = MeCab()

def build_corpus(src_data, tgt_data, tokenize_fn, max_len=40):
    """
    src_data, tgt_data : 소스(질문) / 타겟(답변) 문장 리스트 (병렬 쌍, 같은 인덱스끼리 매칭)
    tokenize_fn        : 토큰화 함수 (여기서는 mecab.morphs)
    max_len             : 허용하는 최대 토큰 개수 (이보다 길면 데이터에서 제외)

    반환값: 토큰화된 (src_corpus, tgt_corpus) - 서로 인덱스가 대응하는 병렬 리스트
    """
    src_corpus, tgt_corpus = [], []

    # 중복 검사를 위한 집합. 소스는 소스끼리, 타겟은 타겟끼리 "따로" 검사합니다.
    # (질문:답변 쌍 전체를 비교하는 게 아니라, 질문 문장 자체가 이미 나온 적 있는지 /
    #  답변 문장 자체가 이미 나온 적 있는지를 각각 확인합니다.)
    seen_src, seen_tgt = set(), set()

    for src_sent, tgt_sent in zip(src_data, tgt_data):
        # 1. 정제: 이미 Step 2에서 적용했지만, 함수 단독으로도 안전하게 동작하도록 한 번 더 적용합니다.
        src_clean = preprocess_sentence(src_sent)
        tgt_clean = preprocess_sentence(tgt_sent)

        # 정제 결과가 빈 문장이면(특수문자로만 이루어졌던 경우) 건너뜁니다.
        if not src_clean or not tgt_clean:
            continue

        # 2. 토큰화: 전달받은 토크나이즈 함수(mecab.morphs)를 사용합니다.
        src_tokens = tokenize_fn(src_clean)
        tgt_tokens = tokenize_fn(tgt_clean)

        # 3. 길이 제한: 토큰 개수가 max_len을 초과하는 문장은 제외합니다.
        if len(src_tokens) > max_len or len(tgt_tokens) > max_len:
            continue

        # 4. 중복 제외: 정제된 "문장 문자열" 기준으로 검사합니다.
        #    소스 또는 타겟 둘 중 하나라도 이미 등장했던 문장이면 이 쌍 전체를 제외해
        #    소스 리스트와 타겟 리스트의 인덱스 대응(병렬 관계)이 흐트러지지 않게 합니다.
        if src_clean in seen_src or tgt_clean in seen_tgt:
            continue

        seen_src.add(src_clean)
        seen_tgt.add(tgt_clean)

        src_corpus.append(src_tokens)
        tgt_corpus.append(tgt_tokens)

    return src_corpus, tgt_corpus

print("슝=3")  # 함수 정의 확인


슝=3


In [21]:
# 토큰 개수 상한 (하이퍼파라미터) - 데이터 분포를 보고 필요시 조정 가능
MAX_LEN = 40

# questions/answers(Step 2에서 정제된 데이터)를 형태소 토큰화 + 길이/중복 필터링합니다.
que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs, max_len=MAX_LEN)

# 필터링 전/후 데이터 개수를 비교해 얼마나 걸러졌는지 확인합니다.
print(f"필터링 전: {len(questions)}개")
print(f"필터링 후: {len(que_corpus)}개")

# 두 코퍼스의 길이가 여전히 일치하는지(병렬 관계가 유지되는지) 확인합니다.
assert len(que_corpus) == len(ans_corpus)

# 토큰화 결과를 샘플로 확인합니다.
for i in range(3):
    print(f"Q(tokens): {que_corpus[i]}")
    print(f"A(tokens): {ans_corpus[i]}")
    print()


필터링 전: 11823개
필터링 후: 7737개
Q(tokens): ['12', '시', '땡', '!']
A(tokens): ['하루', '가', '또', '가', '네요', '.']

Q(tokens): ['1', '지망', '학교', '떨어졌', '어']
A(tokens): ['위로', '해', '드립니다', '.']

Q(tokens): ['3', '박', '4', '일', '놀', '러', '가', '고', '싶', '다']
A(tokens): ['여행', '은', '언제나', '좋', '죠', '.']




## **Step 4. Augmentation (데이터 증강)**
---
사전 훈련된 한국어 Word2Vec 모델([Kyubyong/wordvectors](https://github.com/Kyubyong/wordvectors)의 `Korean (w)`, `ko.bin`)을 이용해
`lexical_sub()`으로 Lexical Substitution을 적용하고, 전체 데이터를 원본의 약 3배로 증강합니다.


In [22]:
# Step 4-1. 사전 훈련된 한국어 Word2Vec 모델(ko.bin) 다운로드
# Kyubyong/wordvectors 저장소의 "Korean (w)" 모델입니다. (Google Drive 파일이라 gdown 사용)
# 파일 용량이 커서(약 77MB) 네트워크 상황에 따라 시간이 걸리거나 실패할 수 있으니,
# 실패 시 https://github.com/Kyubyong/wordvectors 에서 수동으로 받아 같은 폴더에 두어도 됩니다.
!pip install -q gdown
!gdown "https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU" -O ko.zip
!unzip -o -q ko.zip


Downloading...
From (original): https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU
From (redirected): https://drive.google.com/uc?id=0B0ZXk88koS2KbDhXdWg1Q2RydlU&confirm=t&uuid=459d895c-c12b-4df5-897a-09cd1278c384
To: /content/ko.zip
100% 80.6M/80.6M [00:02<00:00, 36.5MB/s]


In [23]:
!pip install gensim

import gensim
from gensim.models import KeyedVectors
import pickle
import numpy as np

# Step 4-2. 다운로드한 Word2Vec 모델을 로드합니다.
# [수정 이유] Kyubyong/wordvectors의 ko.bin은 아주 오래된 gensim(0.x/1.x대) 버전으로 저장되어 있어,
# 최신 gensim(4.x)의 Word2Vec.load()로 곧바로 불러오면 내부 클래스/속성 이름이 바뀐 탓에
# (예: wv.vocab -> wv.key_to_index, wv.index2word -> wv.index_to_key, most_similar 위치 이동 등)
# 로드가 실패하거나 속성 접근 에러가 발생합니다. 아래에서는 이를 단계적으로 처리합니다.
#   1) pickle로 직접 로드 시도 -> 실패하면
#   2) KeyedVectors.load_word2vec_format()으로 재시도
# 로드 후에는 반드시 아래 검증 셀에서 벡터가 정상(0벡터/NaN 아님)인지, most_similar 결과가
# 말이 되는지 눈으로 확인하세요. (pickle 직접 로드는 gensim의 공식 로더를 우회하는 방식이라
# 예외 없이 "로드는 되었지만 벡터가 비어있는" 상황이 생길 수 있습니다.)
try:
    with open('ko.bin', 'rb') as f:
        ko_model = pickle.load(f, encoding='latin1')

    # Legacy Word2Vec 객체 대응 (모델 전체가 아니라 KeyedVectors만 필요)
    wv = ko_model.wv if hasattr(ko_model, 'wv') else ko_model

    # Gensim 4.x 호환성 보정: 과거 속성명을 최신 속성명(key_to_index)으로 매핑
    if not hasattr(wv, 'key_to_index'):
        if hasattr(wv, 'vocab'):
            wv.key_to_index = {word: unit.index for word, unit in wv.vocab.items()}
        elif hasattr(wv, 'index2word'):
            wv.key_to_index = {word: i for i, word in enumerate(wv.index2word)}

    # most_similar 메서드가 없는 경우, raw 벡터(syn0)를 이용해 코사인 유사도를 직접 계산합니다.
    if not hasattr(wv, 'most_similar'):
        def manual_most_similar(word, topn=10):
            if word not in wv.key_to_index:
                raise KeyError(f"Word '{word}' not in vocabulary")

            target_vec = wv.syn0[wv.key_to_index[word]]
            target_vec = target_vec / np.linalg.norm(target_vec)

            all_vecs = wv.syn0 / np.linalg.norm(wv.syn0, axis=1, keepdims=True)
            sims = np.dot(all_vecs, target_vec)

            indices = np.argsort(sims)[::-1]
            result = []
            for i in indices:
                w = wv.index2word[i]
                if w != word:
                    result.append((w, float(sims[i])))
                if len(result) >= topn:
                    break
            return result

        wv.most_similar = manual_most_similar

    print("pickle 직접 로드 + 호환성 패치 성공")

except Exception as e:
    print(f"pickle 로드 실패: {e}. KeyedVectors.load_word2vec_format()으로 재시도합니다...")
    wv = KeyedVectors.load_word2vec_format('ko.bin', binary=True, unicode_errors='ignore')
    print("KeyedVectors 로드 성공")


pickle 직접 로드 + 호환성 패치 성공


In [24]:
# Step 4-2 검증. 임베딩이 실제로 유효한 값을 담고 있는지 반드시 확인합니다.
# (pickle 직접 로드는 gensim 공식 로더를 우회하므로, 예외 없이 "로드는 되었지만
#  벡터가 비어있거나 0으로만 채워진" 상태일 수 있습니다. 아래 검증을 건너뛰지 마세요.)

vecs = wv.syn0 if hasattr(wv, 'syn0') else wv.vectors
print("벡터 shape:", vecs.shape)
print("벡터 norm 평균/최소/최대:", np.linalg.norm(vecs, axis=1).mean(),
      np.linalg.norm(vecs, axis=1).min(), np.linalg.norm(vecs, axis=1).max())
print("NaN 포함 여부:", np.isnan(vecs).any())

# 여러 단어로 most_similar를 확인해, 결과가 실제로 의미상 유사한 단어들인지 눈으로 판단하세요.
for test_word in ["행복", "학교", "음식"]:
    if test_word in wv.key_to_index:
        print(f"'{test_word}'와 유사한 단어: {wv.most_similar(test_word, topn=5)}")
    else:
        print(f"'{test_word}'는 사전에 없습니다.")

# 위 벡터 norm이 전부 0이거나 NaN이 섞여 있다면, 또는 유사 단어 결과가 전혀 관련 없는 단어들로만
# 나온다면 pickle 로드가 "성공은 했지만 실제로는 깨진 상태"인 것입니다.
# 이 경우 Word2Vec.load('ko.bin') 또는 KeyedVectors.load_word2vec_format(...) 등
# 다른 로드 방식을 시도하거나, ko.zip 안에 syn0.npy 등 사이드카 파일이 함께 있는지 확인하세요.


벡터 shape: (30185, 200)
벡터 norm 평균/최소/최대: 8.300879 1.7106013 31.380342
NaN 포함 여부: False
'행복'와 유사한 단어: [('사랑', 0.6759077310562134), ('기쁨', 0.6493781805038452), ('즐거움', 0.6396492719650269), ('삶', 0.6299898028373718), ('젊음', 0.6187379360198975)]
'학교'와 유사한 단어: [('학교의', 0.7560996413230896), ('강습소', 0.7425637245178223), ('중고등학교', 0.7386142015457153), ('전문학교', 0.7356827855110168), ('사립학교', 0.7347193956375122)]
'음식'와 유사한 단어: [('생선', 0.7038266062736511), ('후식', 0.7021129131317139), ('해산물', 0.691323459148407), ('반찬', 0.6886610388755798), ('조리법', 0.6833649277687073)]


In [25]:
import random

def lexical_sub(tokens, wv):
    """
    tokens : 형태소 토큰 리스트
    wv     : Word2Vec 또는 KeyedVectors 객체

    [설명] 사전 훈련된 Word2Vec 임베딩을 이용해, 문장 내 임베딩 어휘에 존재하는
    토큰 중 하나를 무작위로 골라 가장 유사한 단어로 교체합니다. (Lexical Substitution)

    [수정 이유] 이전 버전은 `tok in wv`로 어휘 포함 여부를 검사했는데, 일부 gensim
    버전/패치된 객체에서는 `in` 연산자가 제대로 동작하지 않아(iterable 관련 오류) 실패했습니다.
    `tok in wv.key_to_index`로 명시적으로 딕셔너리 검사하도록 고쳐 버전에 관계없이 안전하게 동작합니다.
    """
    if not hasattr(wv, 'key_to_index'):
        return tokens

    # 임베딩 어휘에 존재하는 토큰만 대체 후보로 고려합니다.
    valid_tokens = [tok for tok in tokens if tok in wv.key_to_index]

    if not valid_tokens:
        return tokens

    selected_tok = random.choice(valid_tokens)

    try:
        similar_words = wv.most_similar(selected_tok, topn=1)
        if similar_words:
            similar_word = similar_words[0][0]
            return [similar_word if tok == selected_tok else tok for tok in tokens]
        return tokens
    except Exception:
        # most_similar가 실패하는 예외적인 경우(예: 벡터 계산 오류)에는 원본을 그대로 반환합니다.
        return tokens

print("슝=3")


슝=3


In [26]:
from tqdm import tqdm
import random
import numpy as np
import torch

# 재현성 (#5)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# [업그레이드 #2] 증강 '전에' 원본을 train/val로 분리합니다.
#  - val은 증강하지 않습니다(누수 방지). 증강 복제본이 train에, 원본이 val에 섞이면 평가가 과대평가됩니다.
VAL_RATIO = 0.1
N = len(que_corpus)
perm = np.random.RandomState(42).permutation(N)
val_size = int(N * VAL_RATIO)
val_ids = set(perm[:val_size].tolist())

train_que = [q for i, q in enumerate(que_corpus) if i not in val_ids]
train_ans = [a for i, a in enumerate(ans_corpus) if i not in val_ids]
val_que   = [q for i, q in enumerate(que_corpus) if i in val_ids]
val_ans   = [a for i, a in enumerate(ans_corpus) if i in val_ids]

# 원본(train) 보존
original_que_corpus = list(train_que)
original_ans_corpus = list(train_ans)

# [업그레이드 #2] 증강은 train 에만 적용
aug_que_corpus = [lexical_sub(q, wv) for q in tqdm(original_que_corpus, desc="질문 Augmentation")]
aug_ans_corpus = [lexical_sub(a, wv) for a in tqdm(original_ans_corpus, desc="답변 Augmentation")]

print("\n[Augmentation Sample]")
for i in range(3):
    print(f"[원본 Q] {original_que_corpus[i]}")
    print(f"[증강 Q] {aug_que_corpus[i]}")
    print()

que_corpus = original_que_corpus + aug_que_corpus + original_que_corpus
ans_corpus = original_ans_corpus + original_ans_corpus + aug_ans_corpus
assert len(que_corpus) == len(ans_corpus)

original_size = len(original_que_corpus)
print(f"증강 전(train) 개수: {original_size}개")
print(f"증강 후(train) 개수: {len(que_corpus)}개 (약 {len(que_corpus)/original_size:.1f}배)")
print(f"검증셋(val) 개수: {len(val_que)}개 (증강 안 함)")


답변 Augmentation: 100%|██████████| 6964/6964 [02:01<00:00, 57.24it/s]


[Augmentation Sample]
[원본 Q] ['12', '시', '땡', '!']
[증강 Q] ['12', '시', '땡', '캐치']

[원본 Q] ['1', '지망', '학교', '떨어졌', '어']
[증강 Q] ['1', '중퇴', '학교', '떨어졌', '어']

[원본 Q] ['3', '박', '4', '일', '놀', '러', '가', '고', '싶', '다']
[증강 Q] ['3', '김', '4', '일', '놀', '러', '가', '고', '싶', '다']

증강 전(train) 개수: 6964개
증강 후(train) 개수: 20892개 (약 3.0배)
검증셋(val) 개수: 773개 (증강 안 함)



## **Step 5. 데이터 벡터화**
---
`ans_corpus`에 `<start>`/`<end>` 토큰을 추가하고, `que_corpus`와 결합해 **하나의 통합 단어 사전**을 구축한 뒤(같은 언어이므로 임베딩 공유),
`enc_train`, `dec_train`으로 벡터화합니다.


In [27]:
# Step 5-1. 타겟(ans_corpus)에 <start>, <end> 토큰 추가 (train + val 동일 적용)
ans_corpus = [["<start>"] + sent + ["<end>"] for sent in ans_corpus]
val_ans = [["<start>"] + sent + ["<end>"] for sent in val_ans]  # [업그레이드 #2]

print(ans_corpus[0])


['<start>', '하루', '가', '또', '가', '네요', '.', '<end>']


In [28]:
from collections import Counter

# Step 5-2. 단어 사전 구축
# 챗봇은 소스(질문)와 타겟(답변)이 "같은 언어"를 사용한다는 점이 번역기와 다른 부분입니다.
# 두 코퍼스를 합쳐 하나의 통합 사전을 만들고, 인코더/디코더가 임베딩을 공유(shared=True)하도록 합니다.

PAD, START, END, UNK = "<pad>", "<start>", "<end>", "<unk>"

# 특수 토큰은 앞쪽 id(0~3)에 고정 배치합니다. (번역기 프로젝트의 SentencePiece pad/bos/eos/unk id 체계와 동일하게 유지)
word2idx = {PAD: 0, START: 1, END: 2, UNK: 3}

# 전체 데이터(질문 + 답변)에 등장하는 토큰 빈도를 계산합니다.
counter = Counter()
for sent in que_corpus + ans_corpus:
    counter.update(sent)

MAX_VOCAB_SIZE = 20000  # 사전에 담을 최대 단어 수 (하이퍼파라미터, 데이터 규모에 맞게 조정 가능)

for word, _ in counter.most_common(MAX_VOCAB_SIZE):
    if word not in word2idx:  # <start>/<end>는 이미 등록되어 있으므로 중복 방지
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}
VOCAB_SIZE = len(word2idx)

print(f"통합 단어 사전 크기: {VOCAB_SIZE}")


통합 단어 사전 크기: 6915


In [29]:
import torch

def tokens_to_ids(tokens, word2idx):
    # 토큰 리스트를 단어 사전 id 시퀀스로 변환. 사전에 없는 단어는 <unk> 처리.
    return [word2idx.get(tok, word2idx[UNK]) for tok in tokens]

def pad_sequences_custom(sequences, max_len, pad_value=0):
    padded = []
    for seq in sequences:
        if len(seq) > max_len:
            seq = seq[:max_len]
        else:
            seq = seq + [pad_value] * (max_len - len(seq))
        padded.append(seq)
    return torch.tensor(padded, dtype=torch.long)

ENC_MAX_LEN = MAX_LEN
DEC_MAX_LEN = MAX_LEN + 2

enc_train = pad_sequences_custom([tokens_to_ids(q, word2idx) for q in que_corpus], max_len=ENC_MAX_LEN)
dec_train = pad_sequences_custom([tokens_to_ids(a, word2idx) for a in ans_corpus], max_len=DEC_MAX_LEN)

# [업그레이드 #2] 검증셋도 동일하게 벡터화 (vocab은 train 기준 -> val 미등장 단어는 자동 <unk>)
enc_val = pad_sequences_custom([tokens_to_ids(q, word2idx) for q in val_que], max_len=ENC_MAX_LEN)
dec_val = pad_sequences_custom([tokens_to_ids(a, word2idx) for a in val_ans], max_len=DEC_MAX_LEN)

print("enc_train:", enc_train.shape, "/ dec_train:", dec_train.shape)
print("enc_val:", enc_val.shape, "/ dec_val:", dec_val.shape)

print("Q(tokens):", que_corpus[0])
print("A(tokens):", ans_corpus[0])


enc_train: torch.Size([20892, 40]) / dec_train: torch.Size([20892, 42])
enc_val: torch.Size([773, 40]) / dec_val: torch.Size([773, 42])
Q(tokens): ['12', '시', '땡', '!']
A(tokens): ['<start>', '하루', '가', '또', '가', '네요', '.', '<end>']



## **Step 6. 훈련하기**
---
17장에서 구현한 Transformer를 그대로 재사용합니다. 다만 데이터가 약 3만 개 내외로 번역기 프로젝트보다 작기 때문에
**모델 크기를 줄이고 dropout을 높이는 등 과적합 방지 위주로 하이퍼파라미터를 조정**했습니다.
(아래 수치는 시작점입니다. 학습 곡선/생성 결과를 보며 직접 조정해 나가야 합니다.)

> 참고: 원본 스캐폴드 코드에는 ① Positional Encoding이 numpy 배열로 반환되어 텐서 연산과 맞지 않는 부분,
> ② Optimizer보다 Scheduler를 먼저 생성해 순서가 꼬인 부분, ③ Loss 계산 시 CrossEntropyLoss가 기대하는
> 차원 순서(N, C, ...)와 실제 텐서 모양이 어긋나는 부분이 있어, 아래 코드에서는 이를 바로잡아 정상 동작하도록 구현했습니다.


In [30]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def positional_encoding(pos, d_model):
    # [설명] 트랜스포머는 RNN과 달리 단어 순서를 순차적으로 받지 않으므로, Sine/Cosine 주기 함수로 위치 정보를 임베딩합니다.
    def cal_angle(position, i):
        return position / np.power(10000, int(i) / d_model)

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    # numpy 배열을 torch 텐서로 변환해 반환합니다. (Transformer 내부에서 .to(device) 등 텐서 연산에 바로 사용하기 위함)
    return torch.tensor(sinusoid_table, dtype=torch.float32)

print("슝=3")


슝=3


In [31]:
class MultiHeadAttention(nn.Module):
    # [설명] Q, K, V 행렬을 분할해 병렬로 Scaled Dot-Product Attention을 수행하여 문장 내 문맥을 동시에 파악합니다.
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = K.shape[-1]
        QK = torch.matmul(Q, K.transpose(-2, -1))
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        if mask is not None:
            scaled_qk = scaled_qk.masked_fill(mask != 0, float('-1e9'))  # [수정] 마스크(1=가릴 위치)가 1인 곳을 -inf 처리

        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)

        return out, attentions

    def split_heads(self, x):
        bsz, seq_len, d_model = x.shape
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask)

        out = self.combine_heads(out)
        out = self.linear(out)

        return out, attention_weights

print("슝=3")


슝=3


In [32]:
class PoswiseFeedForwardNet(nn.Module):
    # [설명] 어텐션 층을 거친 출력값을 처리하는 완전연결 신경망으로, 각 단어 벡터별로 독립적으로 비선형(ReLU) 연산을 수행합니다.
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

print("슝=3")


슝=3


In [33]:
class EncoderLayer(nn.Module):
    # [설명] 인코더 단일 레이어: Multi-Head Self Attention과 Position-wise FFN, 잔차 연결(Residual) 및 Layer Normalization으로 구성됩니다.
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()

        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Self Attention
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.dropout(out)
        out += residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, enc_attn


class DecoderLayer(nn.Module):
    # [설명] 디코더 단일 레이어: 미래 단어를 보지 못하게 하는 Masked Self Attention과 인코더 출력을 참조하는 Encoder-Decoder Attention이 포함됩니다.
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causality_mask, padding_mask):
        # Masked Multi-Head Self Attention (자기 자신, 미래 토큰 마스킹)
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, padding_mask)
        out = self.dropout(out)
        out += residual

        # Encoder-Decoder Multi-Head Attention (인코더 출력을 참조)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, causality_mask)
        out = self.dropout(out)
        out += residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, dec_attn, dec_enc_attn

print("슝=3")


슝=3


In [34]:
class Encoder(nn.Module):
    # [설명] 다수의 인코더 레이어를 쌓아올려 질문 문장 전체의 복잡한 문맥적 특징을 추출합니다.
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)

        # [버그 수정] Pre-LN(Pre-LayerNorm) 구조에서는 각 서브레이어 "입력"에 LayerNorm을 적용하기 때문에,
        # 마지막 레이어를 통과한 뒤에는 정규화되지 않은 residual 누적값이 그대로 출력됩니다.
        # 이 상태로 Decoder의 Encoder-Decoder Attention에 Key/Value로 들어가면 값의 크기(norm)가 커져
        # softmax가 특정 위치에만 집중(attention collapse)하는 문제가 생길 수 있어, 스택 마지막에
        # LayerNorm을 한 번 더 적용합니다. (표준 Pre-LN Transformer 구조)
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)

        out = self.final_norm(out)  # 스택 종료 후 최종 정규화

        return out, enc_attns


class Decoder(nn.Module):
    # [설명] 다수의 디코더 레이어를 쌓아올려 답변 문장을 순차적으로 예측하며 생성해 냅니다.
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

        # [버그 수정] Encoder와 동일한 이유로, 디코더 스택 마지막에도 LayerNorm을 추가합니다.
        # 이게 없으면 fc(출력층)로 들어가는 값의 스케일이 들쭉날쭉해져 학습이 불안정해지고,
        # 생성되는 답변이 몇 가지 패턴으로만 반복되는 모드 붕괴(mode collapse) 현상으로 이어질 수 있습니다.
        self.final_norm = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, enc_out, causality_mask, padding_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](
                out, enc_out, causality_mask, padding_mask
            )
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)

        out = self.final_norm(out)  # 스택 종료 후 최종 정규화

        return out, dec_attns, dec_enc_attns

print("슝=3")


슝=3


In [35]:
class Transformer(nn.Module):
    # [설명] 인코더와 디코더를 결합한 트랜스포머 완성 모델. 입력/타겟을 마스킹 처리하고 최종 예측값과 어텐션 가중치를 반환합니다.
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared=True):
        super(Transformer, self).__init__()

        self.d_model = d_model
        self.shared = shared

        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # positional_encoding()이 이미 torch 텐서를 반환하므로 별도 변환 없이 바로 사용합니다.
        self.pos_encoding = positional_encoding(pos_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        # 챗봇은 소스/타겟이 같은 언어이므로, 출력층 가중치를 디코더 임베딩과 공유(Weight Sharing)합니다.
        if shared:
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        seq_len = x.shape[1]
        out = emb(x)

        # [업그레이드 #5] 임베딩 스케일은 shared 여부와 무관하게 항상 적용
        out = out * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32, device=out.device))

        pos_enc = self.pos_encoding[:seq_len, :].unsqueeze(0).to(x.device)
        out += pos_enc

        out = self.dropout(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, causality_mask, dec_mask):
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        enc_out, enc_attns = self.encoder(enc_in, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in, enc_out, causality_mask, dec_mask)

        logits = self.fc(dec_out)

        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")


슝=3


In [36]:
def generate_padding_mask(seq):
    """ 패딩된 부분(0)을 1로 변환하여 마스크 생성 """
    mask = (seq == 0).float()
    return mask[:, None, None, :]

def generate_causality_mask(src_len, tgt_len):
    """ 미래 정보를 참조하지 않도록 Causal(순차) Mask 생성 """
    mask = 1 - torch.cumsum(torch.eye(src_len, tgt_len), dim=0)
    return mask.float()

def generate_masks(src, tgt):
    """ Encoder / Decoder에서 사용할 마스크를 한 번에 생성합니다. """
    enc_mask = generate_padding_mask(src)
    dec_mask = generate_padding_mask(tgt)

    # 디코더 자기 자신(self-attention)에는 패딩 마스크 + causal 마스크를 함께 적용합니다.
    dec_causality_mask = generate_causality_mask(tgt.shape[1], tgt.shape[1])
    dec_mask = torch.max(dec_mask, dec_causality_mask.to(dec_mask.device))

    # [수정] 인코더-디코더(cross) attention에는 인코더 '패딩 마스크만' 적용합니다.
    #        디코더는 매 시점 소스(질문) 문장 전체를 참조해야 하므로 causal 마스크를 걸면 안 됩니다.
    dec_enc_mask = generate_padding_mask(src)

    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")


슝=3


In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 디바이스:", device)

# 하이퍼파라미터 (데이터가 약 3만 개로 번역기 프로젝트보다 작으므로,
# 과적합을 피하기 위해 레이어 수 / 모델 크기를 줄이고 dropout을 높였습니다. 필요시 조정하세요.)
N_LAYERS = 2
D_MODEL = 256
N_HEADS = 8
D_FF = 512
DROPOUT = 0.3
POS_LEN = max(ENC_MAX_LEN, DEC_MAX_LEN) + 10  # 위치 인코딩 테이블 여유분

transformer = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,   # 챗봇은 소스/타겟이 같은 언어이므로 동일한 통합 사전을 사용합니다.
    pos_len=POS_LEN,
    dropout=DROPOUT,
    shared=True                  # 인코더/디코더 임베딩 및 출력층 가중치 공유
).to(device)

# 학습 가능한 파라미터 수를 확인합니다. (모델 크기 가늠용)
n_params = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
print(f"학습 가능한 파라미터 수: {n_params:,}")


사용 디바이스: cuda
학습 가능한 파라미터 수: 6,184,195


In [38]:
class LearningRateScheduler(torch.optim.lr_scheduler._LRScheduler):
    # [설명] 초반에는 학습률을 선형 증가(warmup)시키고 이후 점진적으로 감소(decay)시켜 학습 안정성을 높이는 커스텀 스케줄러입니다.
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super(LearningRateScheduler, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        step = max(1, self.last_epoch)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        lr = (self.d_model ** -0.5) * min(arg1, arg2)
        return [lr for _ in self.base_lrs]

# 1) Optimizer를 먼저 생성합니다. (스케줄러가 optimizer를 감싸는 형태이므로 이 순서가 중요합니다)
optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

# 2) Warmup + Decay 학습률 스케줄러를 정의합니다. (Warmup Steps는 데이터가 작은 만큼 번역기 프로젝트보다 짧게 설정)
WARMUP_STEPS = 1000
scheduler = LearningRateScheduler(optimizer, d_model=D_MODEL, warmup_steps=WARMUP_STEPS)

print("슝=3")


슝=3


In [39]:
loss_object = torch.nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1)  # [업그레이드 #4]

def loss_function(real, pred):
    # [설명] 교차 엔트로피 손실을 계산하되, 패딩(0) 토큰은 손실 계산에서 제외(마스킹)합니다.
    # CrossEntropyLoss는 (N, C, ...) 형태를 기대하므로 (batch, seq_len, vocab) -> (batch, vocab, seq_len)로 축을 바꿔줍니다.
    mask = (real != 0).float()

    loss_ = loss_object(pred.transpose(1, 2), real)
    loss_ *= mask

    # 마스킹되지 않은(=패딩이 아닌) 토큰 개수로 나누어 평균 손실을 구합니다.
    return loss_.sum() / mask.sum()

print("슝=3")


슝=3


In [40]:
def train_step(src, tgt, model, optimizer, scheduler):
    # [설명] 1회 배치 학습. teacher forcing: 디코더 입력=전체 타겟, 정답=한 칸 밀린 토큰.
    gold = tgt[:, 1:]
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt)

    optimizer.zero_grad()
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt, enc_mask, dec_enc_mask, dec_mask)
    loss = loss_function(gold, predictions[:, :-1])

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # [업그레이드 #4] gradient clipping
    optimizer.step()
    scheduler.step()

    return loss, enc_attns, dec_attns, dec_enc_attns


@torch.no_grad()
def eval_loss(enc_data, dec_data, model, batch_size=128):
    # [업그레이드 #2] 검증셋 평균 손실(grad 없음) - best 체크포인트/early stopping 기준
    model.eval()
    total, n = 0.0, 0
    for i in range(0, enc_data.shape[0], batch_size):
        s = enc_data[i:i + batch_size]
        t = dec_data[i:i + batch_size]
        enc_mask, dec_enc_mask, dec_mask = generate_masks(s, t)
        pred, *_ = model(s, t, enc_mask, dec_enc_mask, dec_mask)
        l = loss_function(t[:, 1:], pred[:, :-1])
        total += l.item() * s.shape[0]
        n += s.shape[0]
    model.train()
    return total / max(n, 1)

print("슝=3")


슝=3


In [41]:
def evaluate(sentence, model, word2idx, idx2word, max_len, rep_penalty=1.3):
    # [설명] 질문을 입력해 답변 생성 (그리디 디코딩 + 반복 억제)
    model.eval()

    clean = preprocess_sentence(sentence)
    tokens = mecab.morphs(clean)

    input_ids = tokens_to_ids(tokens, word2idx)
    enc_input = torch.tensor([input_ids], dtype=torch.long).to(device)
    dec_input = torch.tensor([[word2idx[START]]], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            enc_mask, dec_enc_mask, dec_mask = generate_masks(enc_input, dec_input)
            predictions, enc_attns, dec_attns, dec_enc_attns = model(
                enc_input, dec_input, enc_mask, dec_enc_mask, dec_mask
            )
            logits = predictions[0, -1].clone()
            generated = dec_input[0, 1:].tolist()
            # [업그레이드 #1] 반복 페널티(soft) + no-repeat bigram(hard): "좋 은 좋 은..." 반복 방지
            for tk in set(generated):
                logits[tk] /= rep_penalty
            if generated:
                prev = generated[-1]
                for a, b in zip(generated, generated[1:]):
                    if a == prev:
                        logits[b] = float('-inf')
            predicted_id = torch.argmax(logits).item()

            if predicted_id == word2idx[END]:
                break

            dec_input = torch.cat(
                [dec_input, torch.tensor([[predicted_id]], device=device)], dim=-1
            )

    model.train()

    result_ids = dec_input[0, 1:].tolist()
    result_tokens = [idx2word.get(i, UNK) for i in result_ids]
    return " ".join(result_tokens)


def translate(sentence, model, word2idx, idx2word, max_len):
    result = evaluate(sentence, model, word2idx, idx2word, max_len)
    print(f"Q: {sentence}")
    print(f"A: {result}")
    return result

print("슝=3")


슝=3



### **💾 체크포인트 (런타임 끊김 대비)**
---
Colab은 런타임이 예고 없이 종료될 수 있으므로, **매 epoch이 끝날 때마다 모델/옵티마이저 상태를 저장**하고,
학습을 다시 시작할 때 저장된 지점부터 이어서 진행하도록 합니다. 저장 위치는 런타임 디스크가 아닌
**Google Drive**로 설정해, 런타임이 초기화되어도 파일이 남아있도록 합니다.


In [42]:
import os

# 체크포인트를 Google Drive에 저장할지 여부 (True 권장 - 런타임이 리셋되어도 파일이 남습니다)
USE_DRIVE = True

# [변경] 새 체크포인트 경로로 처음부터 시작하고 싶을 때는 이 값만 바꾸세요.
#   - 새 이름을 쓰면 불러올 파일이 없어 자동으로 epoch 1부터 새로 학습합니다.
#   - 예전 파일(예: v2)은 그대로 남아 백업이 됩니다. 또 새로 하려면 'v4', 'v5'...로 올리면 됩니다.
RUN_NAME = 'v3'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/chatbot_checkpoint'
else:
    # 로컬(런타임) 디스크에 저장 - 런타임이 끊기면 함께 사라지므로 재개용으로는 권장하지 않습니다.
    CKPT_DIR = '/content/chatbot_checkpoint'

os.makedirs(CKPT_DIR, exist_ok=True)

CKPT_PATH = os.path.join(CKPT_DIR, f'chatbot_{RUN_NAME}_latest.pt')     # 매 epoch 저장되는 "최신"(재개용)
CKPT_BEST_PATH = os.path.join(CKPT_DIR, f'chatbot_{RUN_NAME}_best.pt')  # val loss가 가장 낮았던 best

print("체크포인트 저장 경로:", CKPT_DIR)
print("  최신:", os.path.basename(CKPT_PATH))
print("  best:", os.path.basename(CKPT_BEST_PATH))


Mounted at /content/drive
체크포인트 저장 경로: /content/drive/MyDrive/chatbot_checkpoint
  최신: chatbot_v3_latest.pt
  best: chatbot_v3_best.pt


In [43]:
def save_checkpoint(path, epoch, model, optimizer, scheduler, loss, word2idx, idx2word, hyperparams):
    """
    [설명] 학습을 이어가는 데 필요한 모든 상태를 저장합니다.
    - model/optimizer/scheduler 상태: 이어서 학습하기 위함
    - word2idx/idx2word, hyperparams: 런타임이 완전히 초기화되어 처음부터 다시 실행하더라도
      동일한 모델 구조/사전으로 복원할 수 있도록 함께 저장합니다.
    """
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_last_epoch': scheduler.last_epoch,
        'loss': loss,
        'word2idx': word2idx,
        'idx2word': idx2word,
        'hyperparams': hyperparams,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler):
    """
    [설명] 저장된 체크포인트를 불러와 model/optimizer/scheduler 상태를 복원하고,
    이어서 학습을 시작할 epoch 번호를 반환합니다. 체크포인트가 없으면 0(처음부터)을 반환합니다.

    [수정 이유] 다음 두 경우에 예외로 학습이 통째로 멈추던 문제를 고쳤습니다.
    1. VOCAB_SIZE 불일치: Step 4 증강(lexical_sub)이 random.choice로 매번 다른 단어를 치환하기
       때문에, 시드를 고정하지 않으면 실행할 때마다 단어 사전 크기가 미세하게 달라질 수 있습니다.
       (예: 7243 vs 7237) 이러면 임베딩 크기가 달라 체크포인트를 이어서 쓸 수 없습니다.
    2. 모델 구조 변경: Encoder/Decoder에 final_norm을 추가하는 등 모델 구조를 바꾸면,
       이전에 저장된 체크포인트에는 해당 파라미터가 아예 없어 strict 로드가 실패합니다.
    두 경우 모두 "이어서 학습 불가능한 상태"이므로, 에러를 내며 멈추는 대신 경고만 출력하고
    처음(epoch 0)부터 새로 학습을 시작하도록 합니다.
    """
    if not os.path.exists(path):
        print("저장된 체크포인트가 없습니다. 처음부터 학습을 시작합니다.")
        return 0

    ckpt = torch.load(path, map_location=device)

    # 1) VOCAB_SIZE가 다르면 임베딩 크기 자체가 다르므로 애초에 로드를 시도하지 않습니다.
    ckpt_vocab_size = ckpt.get('hyperparams', {}).get('VOCAB_SIZE')
    if ckpt_vocab_size != VOCAB_SIZE:
        print(f"경고: 체크포인트의 VOCAB_SIZE({ckpt_vocab_size})가 현재 설정({VOCAB_SIZE})과 다릅니다.")
        print("-> Step 4 증강에 랜덤 시드가 고정되지 않았거나 데이터가 바뀐 것이 원인일 수 있습니다.")
        print("-> 이 체크포인트는 이어서 사용할 수 없어 처음부터 학습을 시작합니다.")
        return 0

    # 2) 모델 구조가 바뀌어 state_dict의 key가 맞지 않는 경우도 에러 대신 처음부터 시작하도록 처리합니다.
    try:
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.last_epoch = ckpt['scheduler_last_epoch']
    except RuntimeError as e:
        print("체크포인트 로드 실패 (모델 구조가 바뀐 것으로 보입니다):")
        print(e)
        print("-> 처음부터 학습을 시작합니다. 구버전 체크포인트가 필요 없다면 Drive에서 파일을 지우거나")
        print("   CKPT_PATH/CKPT_BEST_PATH 파일명을 새로 바꿔 사용하세요.")
        return 0

    start_epoch = ckpt['epoch'] + 1
    print(f"체크포인트 로드 완료 (epoch {ckpt['epoch'] + 1}, loss={ckpt['loss']:.4f}) -> epoch {start_epoch + 1}부터 재개합니다.")
    return start_epoch

print("슝=3")


슝=3


In [44]:
from tqdm import tqdm
import random

# 프로젝트에서 요구하는 4개의 테스트 질문
test_questions = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

BATCH_SIZE = 64
EPOCHS = 40          # early stopping이 있으니 넉넉히 설정
SAVE_EVERY = 1
PATIENCE = 5         # [업그레이드 #2] val loss가 PATIENCE epoch 연속 개선 없으면 조기 종료

hyperparams = {
    'N_LAYERS': N_LAYERS, 'D_MODEL': D_MODEL, 'N_HEADS': N_HEADS,
    'D_FF': D_FF, 'DROPOUT': DROPOUT, 'POS_LEN': POS_LEN, 'VOCAB_SIZE': VOCAB_SIZE,
}

enc_train = enc_train.to(device)
dec_train = dec_train.to(device)
enc_val = enc_val.to(device)   # [업그레이드 #2]
dec_val = dec_val.to(device)

start_epoch = load_checkpoint(CKPT_PATH, transformer, optimizer, scheduler)
best_loss = float('inf')
bad_epochs = 0

if os.path.exists(CKPT_BEST_PATH):
    best_ckpt = torch.load(CKPT_BEST_PATH, map_location=device)
    best_loss = best_ckpt['loss']
    print(f"기존 best(val) loss: {best_loss:.4f}")

for epoch in range(start_epoch, EPOCHS):
    total_loss = 0
    # [업그레이드 #3] 매 epoch 전체 예시를 새로 셔플 (배치 시작 위치만 섞던 문제 해결)
    perm = torch.randperm(enc_train.shape[0], device=enc_train.device)
    n_batches = (enc_train.shape[0] + BATCH_SIZE - 1) // BATCH_SIZE
    t = tqdm(range(n_batches))

    for batch in t:
        b = perm[batch * BATCH_SIZE:(batch + 1) * BATCH_SIZE]
        batch_loss, enc_attns, dec_attns, dec_enc_attns = train_step(
            enc_train[b], dec_train[b], transformer, optimizer, scheduler
        )
        total_loss += batch_loss.item()
        t.set_description(f"Epoch {epoch + 1}")
        t.set_postfix({"Loss": f"{total_loss / (batch + 1):.4f}"})

    train_loss = total_loss / n_batches
    val_loss = eval_loss(enc_val, dec_val, transformer)   # [업그레이드 #2]

    # 정성 평가 (매 epoch 테스트 질문 답변 확인)
    print(f"\n===== Epoch {epoch + 1} (train={train_loss:.4f}, val={val_loss:.4f}) =====")
    for q in test_questions:
        translate(q, transformer, word2idx, idx2word, max_len=DEC_MAX_LEN)
    print()

    # 1) 최신 체크포인트(재개용) - 기준값으로 val_loss 기록
    if (epoch + 1) % SAVE_EVERY == 0:
        save_checkpoint(CKPT_PATH, epoch, transformer, optimizer, scheduler, val_loss, word2idx, idx2word, hyperparams)
        print(f"[체크포인트] 최신 저장 (epoch {epoch + 1})")

    # 2) [업그레이드 #2] best는 'val' loss 기준 + early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        bad_epochs = 0
        save_checkpoint(CKPT_BEST_PATH, epoch, transformer, optimizer, scheduler, val_loss, word2idx, idx2word, hyperparams)
        print(f"[체크포인트] best 갱신 -> 저장 (epoch {epoch + 1}, val={val_loss:.4f})")
    else:
        bad_epochs += 1
        print(f"  (val 개선 없음 {bad_epochs}/{PATIENCE})")
        if bad_epochs >= PATIENCE:
            print(f"조기 종료: val loss가 {PATIENCE} epoch 연속 개선되지 않았습니다. (best val={best_loss:.4f})")
            break

# 제출: best val 시점의 하이퍼파라미터와 그때의 4개 답변을 기록하세요.
# (best 체크포인트 = 검증 성능이 가장 좋았던 epoch. train loss 최소가 아니라 val 기준이라 과적합 이전 지점을 고릅니다.)


저장된 체크포인트가 없습니다. 처음부터 학습을 시작합니다.


Epoch 1: 100%|██████████| 327/327 [00:13<00:00, 23.56it/s, Loss=86.5059]



===== Epoch 1 (train=86.5059, val=10.3822) =====
Q: 지루하다, 놀러가고 싶어.
A: .
Q: 오늘 일찍 일어났더니 피곤하다.
A: .
Q: 간만에 여자친구랑 데이트 하기로 했어.
A: .
Q: 집에 있는다는 소리야.
A: .

[체크포인트] 최신 저장 (epoch 1)
[체크포인트] best 갱신 -> 저장 (epoch 1, val=10.3822)


Epoch 2: 100%|██████████| 327/327 [00:13<00:00, 25.04it/s, Loss=10.4814]



===== Epoch 2 (train=10.4814, val=5.8600) =====
Q: 지루하다, 놀러가고 싶어.
A: .
Q: 오늘 일찍 일어났더니 피곤하다.
A: .
Q: 간만에 여자친구랑 데이트 하기로 했어.
A: .
Q: 집에 있는다는 소리야.
A: .

[체크포인트] 최신 저장 (epoch 2)
[체크포인트] best 갱신 -> 저장 (epoch 2, val=5.8600)


...


===== Epoch 19 (train=3.8060, val=4.2916) =====
Q: 지루하다, 놀러가고 싶어.
A: 좋 은 선택 이 네요 .
Q: 오늘 일찍 일어났더니 피곤하다.
A: 좋 은 선택 이 에요 .
Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 좋 은 생각 이 네요 .
Q: 집에 있는다는 소리야.
A: 좋 은 생각 해 보 세요 .

[체크포인트] 최신 저장 (epoch 19)
  (val 개선 없음 3/5)


Epoch 20: 100%|██████████| 327/327 [00:13<00:00, 23.42it/s, Loss=3.7641]



===== Epoch 20 (train=3.7641, val=4.3067) =====
Q: 지루하다, 놀러가고 싶어.
A: 좋 은 사람 이 네요 .
Q: 오늘 일찍 일어났더니 피곤하다.
A: 좋 은 선택 이 에요 .
Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 생각 을 하 고 있 나 봐요 .
Q: 집에 있는다는 소리야.
A: 생각 하 지 않 겠 어요 .

[체크포인트] 최신 저장 (epoch 20)
  (val 개선 없음 4/5)


Epoch 21: 100%|██████████| 327/327 [00:13<00:00, 23.46it/s, Loss=3.7201]



===== Epoch 21 (train=3.7201, val=4.3049) =====
Q: 지루하다, 놀러가고 싶어.
A: 좋 은 선택 이 네요 .
Q: 오늘 일찍 일어났더니 피곤하다.
A: 좋 은 선택 이 에요 .
Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 생각 을 하 는 것 같 아요 .
Q: 집에 있는다는 소리야.
A: 좋 은 곳 이 네요 .

[체크포인트] 최신 저장 (epoch 21)
  (val 개선 없음 5/5)
조기 종료: val loss가 5 epoch 연속 개선되지 않았습니다. (best val=4.2860)



## **Step 7. 성능 측정하기**
---
정성적 평가(테스트 예문 4개 답변 - Step 6 학습 루프에서 이미 매 epoch 확인)에 더해,
`calculate_bleu()`로 정량적 평가(BLEU Score)를 수행합니다.

> 주의: 챗봇 답변은 정답이 하나로 고정되지 않는 특성상 BLEU가 낮게 나올 수 있습니다.
> 절대적인 성능 지표가 아니라 참고 지표로 활용하세요.


In [45]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(reference_tokens, candidate_tokens):
    """
    reference_tokens : 정답 답변의 형태소 토큰 리스트
    candidate_tokens : 모델이 생성한 답변의 형태소 토큰 리스트

    [설명] NLTK의 sentence_bleu로 BLEU Score를 계산합니다. 문장이 짧을 때 BLEU가
    0점으로 튀는 문제를 막기 위해 SmoothingFunction(method4)을 적용합니다.
    """
    smoothie = SmoothingFunction().method4
    return sentence_bleu([reference_tokens], candidate_tokens, smoothing_function=smoothie)

print("슝=3")


슝=3


In [46]:
def evaluate_tokens(tokens, model, word2idx, idx2word, max_len, rep_penalty=1.3):
    # [설명] evaluate()와 동일한 반복 억제 그리디 디코딩이지만, 이미 토큰화된 입력을 그대로 사용합니다.
    model.eval()

    input_ids = tokens_to_ids(tokens, word2idx)
    enc_input = torch.tensor([input_ids], dtype=torch.long).to(device)
    dec_input = torch.tensor([[word2idx[START]]], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            enc_mask, dec_enc_mask, dec_mask = generate_masks(enc_input, dec_input)
            predictions, _, _, _ = model(enc_input, dec_input, enc_mask, dec_enc_mask, dec_mask)
            logits = predictions[0, -1].clone()
            generated = dec_input[0, 1:].tolist()
            for tk in set(generated):
                logits[tk] /= rep_penalty
            if generated:
                prev = generated[-1]
                for a, b in zip(generated, generated[1:]):
                    if a == prev:
                        logits[b] = float('-inf')
            predicted_id = torch.argmax(logits).item()

            if predicted_id == word2idx[END]:
                break

            dec_input = torch.cat(
                [dec_input, torch.tensor([[predicted_id]], device=device)], dim=-1
            )

    model.train()
    result_ids = dec_input[0, 1:].tolist()
    return [idx2word.get(i, UNK) for i in result_ids]

print("슝=3")


슝=3


In [47]:
import random

# Step 7-1. 정량 평가: [업그레이드 #2] 학습에 쓰지 않은 검증셋(val)으로 BLEU를 계산합니다.
# (기존에는 학습에 사용한 원본으로 측정 -> 과대평가. val 기준이 더 정직한 지표입니다.)
N_EVAL_SAMPLES = 20
eval_indices = random.sample(range(len(val_que)), min(N_EVAL_SAMPLES, len(val_que)))

bleu_scores = []
sample_outputs = []

for idx in eval_indices:
    question_tokens = val_que[idx]
    # val_ans 에는 <start>/<end> 가 포함되어 있으므로 제외하고 정답으로 사용
    reference_tokens = [tok for tok in val_ans[idx] if tok not in (START, END)]

    candidate_tokens = evaluate_tokens(question_tokens, transformer, word2idx, idx2word, max_len=DEC_MAX_LEN)

    score = calculate_bleu(reference_tokens, candidate_tokens)
    bleu_scores.append(score)
    sample_outputs.append((question_tokens, reference_tokens, candidate_tokens, score))

avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"검증셋 샘플 {len(bleu_scores)}개 평균 BLEU Score: {avg_bleu:.4f}\n")

for q, ref, cand, score in sample_outputs[:5]:
    print(f"Q   : {' '.join(q)}")
    print(f"정답: {' '.join(ref)}")
    print(f"생성: {' '.join(cand)}")
    print(f"BLEU: {score:.4f}")
    print()


검증셋 샘플 20개 평균 BLEU Score: 0.0489

Q   : 이제 누구 만나 는 것 도 지치 네
정답: 새로운 만남 은 언제나 어려운 것 같 아요 .
생성: 마음 이 복잡 하 는 것 같 아요 .
BLEU: 0.2985

Q   : 썸 타 는데 맨날 늦 어
정답: 기본 적 인 시간 약속 을 우습 게 여기 는 사람 은 만나 지 않 는 게 좋 을 거 같 아요 .
생성: 썸 이 좋 아 하 는 것 같 아요 .
BLEU: 0.0398

Q   : 있 는 그대로 의 나 를 사랑 하 기
정답: 멋진 사람 이 자 멋진 말 이 네요 .
생성: 사랑 은 항상 항상 시작 해 보 세요 .
BLEU: 0.0257

Q   : 아침 에 일어나 는 거 힘들 다
정답: 조금 더 일찍 잠 을 자 보 세요 .
생성: 더 힘들 겠 네요 .
BLEU: 0.0244

Q   : 나 너무 소심 해
정답: 꼼꼼 한 거 예요 .
생성: 저 도 하 고 싶 어요 .
BLEU: 0.0324



---
### **제출 정리**
---
아래 항목을 정리해서 제출합니다. (루브릭 3개 항목과 매칭)

1. **전처리/증강 결과**: Step 3~4에서 확인한 필터링 전/후 개수, 증강 후 최종 데이터 개수 (약 3만개 목표)
2. **최종 하이퍼파라미터**: Step 6에서 답변이 가장 그럴듯했던 epoch의 `N_LAYERS`, `D_MODEL`, `N_HEADS`, `D_FF`, `DROPOUT`, `WARMUP_STEPS`, `BATCH_SIZE`, `EPOCH`
3. **테스트 예문 4개에 대한 답변**: Step 6 학습 루프 출력 중 가장 좋았던 epoch의 결과
4. **BLEU Score**: 위 Step 7에서 계산한 평균 BLEU Score (+ 참고용 샘플 비교)


---
### **적용된 업그레이드**  
---
1. **디코딩 반복 억제**: 반복 페널티 + no-repeat bigram으로 "좋 은 좋 은…" 반복 제거
2. **검증셋 분리**: 증강 전 원본을 train/val(9:1)로 분리(val 미증강), val loss 기준 best 저장 + early stopping, BLEU도 val로 측정
3. **매 epoch 전체 셔플**: torch.randperm으로 예시 단위 셔플
4. **학습 안정화**: gradient clipping(1.0) + label smoothing(0.1)
5. **기타**: torch 시드 고정, 임베딩 스케일 항상 적용  


---
### **회고**  
---
**배운 점**  
1. teacher forcing의 함정: casual 마스크가 뒤집히면 학습 땐 정답(미래)을 훔쳐봐서 train loss는 잘 떨어지는데 실제 생성은 붕괴한다 → "loss가 낮다 ≠ 잘 된다"  
2. cross-attention에는 causal 마스크를 걸면 안됨(디코더는 소스 문장 전체를 봐야 함).  
3. 번역기와 챗봇이 같은 Seq2Seq지만, 소스=타겟이 같은 언어라 임베딩·출력층 가중치를 공유(weight trying)할 수 있다는 점.  
4. 평가 지표의 정직성. 학습 데이터로 잰 BLEU(0.13)는 과대평가였고, held-out 검증셋으로 재니 0.05 수준 - 지표는 반드시 안본 데이터로.  

**어려웠던 점**  
1. 겉보기엔 정상(loss 하강)인데 답변이 이상해서 원인을 찾기 어려웠던 점 → 작은 softmax 시물레이션으로 "attention이 패딩에 쏠린다"를 눈으로 확인해 마스크 부호 반전을 특정.
2. 사전학습 Word2Vec(ko.bin)이 구버전 gensim 포맷이라 로딩 호환성 문제 → 검증셀로 벡터가 깨지지 않았는지 확인 후 진행.
3. Colab 세션 끊김 대비 체크포인트 저장/복구 그리고 VOCAB_SIZE 불일치 가드로 재개 안정화.  

**아쉬운 점 / 한계**  
1. 데이터가 약 7천 쌍으로 작아 과적합이 빠르게 옴(train은 계속 내려가는데 val은 epoch 12부터 정체). 증강(lexical substitution)이 어색한 쌍을 만들기도 함.
2. label smoothing·val 기준 early stoppig 같은 정석 기법이 이 과제(개방형 챗봇)에선 역효과가 나타남. 답을 밋밋하게("좋은 선택이에요" 반복) 만들고, val 교차엔트로피가 체감 답변 품질과 잘 안 맞아 좋아지기 전에 멈춤.
3. 그리디 디코딩 한계가 존재함. 반복은 억제했지만 답이 짧고 일반적이며, 정답이 하나가 아니라 BLEU가 본질적으로 낮음.  

**다음에 시도할 것**  
1. 디코딩을 beam search 또는 top-k/top-p 샘플링으로 바꿔 다양성·자연스러움 확보.  
2. 소규모 데이터 한계를 사전학습 모델(KoGPT·KoBART) fine-tunning으로 극복.
3. 평가를 BLEU 단독이 아니라 정성평가 + 임배딩 기반 유사도(BERTScore) 병행.
4. dropout·모델 크기·증강 비율을 val 곡석 보며 체계적으로 탐색.